## B_01 · Calidad y limpieza (issue #9)
Perfil de calidad de las 6 carreras reales. Nunca se muestran nombres ni dorsales: siempre se pasa por `quitar_identificadores()` o solo se muestran recuentos.

### Preparar

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
from src.config import DIR_INTERIM
from src.proteccion import quitar_identificadores

### Cargar las 6 carreras tal cual vienen
Se leen los CSV de `datos/interim/` que genera `02_carreras.ipynb` (no están en el repo porque llevan nombres).

In [2]:
carreras = {}
for ruta in sorted(DIR_INTERIM.glob("carrera_0?.csv")):
    carreras[ruta.stem] = pd.read_csv(ruta, dtype=str, keep_default_na=False)

for nombre, df in carreras.items():
    print(nombre, df.shape)

carrera_01 (1489, 7)
carrera_02 (929, 7)
carrera_03 (795, 7)
carrera_04 (875, 7)
carrera_05 (659, 7)
carrera_06 (643, 7)


### Primer vistazo, siempre sin nombres

In [3]:
quitar_identificadores(carreras["carrera_01"]).head()

,posto,posto_sexo,posto_categoria,tempo
0,1,M-1,SNRM-1,00:20:30
1,2,M-2,SNRM-2,00:21:00
2,3,M-3,SUB23M-1,00:21:01
3,4,M-4,SNRM-3,00:21:19
4,5,M-5,MS1M-1,00:21:26


## Perfil de calidad
Para cada carrera se cuentan los problemas de calidad, agrupados por dimensión:
- **Completitud**: vacíos, tiempos que faltan.
- **Validez**: formatos raros (`M-3`, `MS2M-1`, tiempos), categorías desconocidas.
- **Consistencia**: el sexo de "Puesto Sexo" coincide con el de la categoría.
- **Exactitud**: tiempos imposibles (por debajo de 2:30 min/km o más de 1 h).
- **Unicidad**: postos, dorsales o filas repetidas.

La tabla solo tiene recuentos: no aparece ningún dato personal.

In [4]:
from src.config import DIR_EXTERNOS
from src.limpieza import tiempo_a_segundos, marcar_tiempo, parsear_posto_sexo, parsear_posto_categoria

# Distancia de cada carrera (para el tiempo mínimo) y códigos de categoría válidos
distancias = pd.read_csv(DIR_EXTERNOS / "distancias.csv").set_index("carrera_id")["distancia_m"]
categorias_validas = set(pd.read_csv(DIR_EXTERNOS / "categorias.csv")["codigo_carrera"]) - {"?"}


def perfil(df, carrera_id):
    """Cuenta los problemas de calidad de una carrera. Solo devuelve números, nunca nombres."""
    # Separar las columnas mezcladas con las funciones de src/limpieza.py
    sexo_ps = df["posto_sexo"].map(lambda t: parsear_posto_sexo(t)[0])
    cat = df["posto_categoria"].map(parsear_posto_categoria)
    cat_codigo = cat.map(lambda t: t[0])
    cat_sexo = cat.map(lambda t: t[1])
    segundos = df["tempo"].map(tiempo_a_segundos)
    marca = segundos.map(lambda s: marcar_tiempo(s, distancias.get(carrera_id)))
    
    # Filas sin categoría: vienen como "NO-1", "NO-2"... y con posto_sexo "—"
    es_no = df["posto_categoria"].str.fullmatch(r"NO-\d+")


    return {
        "carrera_id": carrera_id,
        "filas": len(df),
        # Completitud
        "vacios": int((df == "").sum().sum()),
        # Validez
        "posto_no_numerico": int((~df["posto"].str.fullmatch(r"\d+")).sum()),
                "posto_sexo_mal_formato": int((sexo_ps.isna() & ~es_no).sum()),
        "posto_categoria_mal_formato": int((cat_codigo.isna() & ~es_no).sum()),
        "categoria_NO": int(es_no.sum()),
        "categoria_ESP": int((cat_codigo == "ESP").sum()),
        "categoria_desconocida": int((cat_codigo.notna() & ~cat_codigo.isin(categorias_validas)).sum()),
        # Consistencia: el sexo de M-3 debe coincidir con el de MS2M-1
        "sexo_no_coincide": int((cat_sexo.notna() & sexo_ps.notna() & (cat_sexo != sexo_ps)).sum()),
        # Completitud y exactitud de los tiempos
        "tempo_mal_formato": int((segundos.isna() & (df["tempo"] != "")).sum()),
        "tempo_faltante": int((marca == "faltante").sum()),
        "tempo_imposible": int((marca == "imposible").sum()),
        # Unicidad
        "posto_repetido": int(df["posto"].duplicated().sum()),
        "dorsal_repetido": int(df["dorsal"].duplicated().sum()),
        "filas_duplicadas": int(df.duplicated().sum()),
    }

### Perfil de las 6 carreras
Una columna por carrera y una fila por problema (`.T` gira la tabla para leerla mejor).

In [5]:
tabla_perfil = pd.DataFrame([perfil(df, nombre) for nombre, df in carreras.items()])
tabla_perfil.set_index("carrera_id").T

carrera_id,carrera_01,carrera_02,carrera_03,carrera_04,carrera_05,carrera_06
filas,1489,929,795,875,659,643
vacios,0,0,0,0,0,0
posto_no_numerico,0,0,0,0,0,0
posto_sexo_mal_formato,0,0,0,0,0,0
posto_categoria_mal_formato,0,0,0,0,0,0
categoria_NO,3,6,5,0,3,0
categoria_ESP,1,0,0,0,1,1
categoria_desconocida,0,0,0,0,0,0
sexo_no_coincide,0,0,0,0,0,0
tempo_mal_formato,0,0,0,0,0,0


### Guardar el perfil
Solo tiene recuentos, así que se guarda en `docs/` y se puede subir al repo.

In [6]:
from src.config import DIR_DOCS

tabla_perfil.to_csv(DIR_DOCS / "perfil_calidad.csv", index=False)
print("Guardado en", (DIR_DOCS / "perfil_calidad.csv").relative_to(RAIZ))

Guardado en docs/perfil_calidad.csv


### Valores raros
Filas que no encajan con el formato `M-3` / `MS2M-1`. Solo se muestran columnas sin datos personales.

In [7]:
# Filas cuyo posto_sexo no tiene el formato M-3 (solo columnas sin datos personales)
columnas_seguras = ["posto", "posto_sexo", "posto_categoria", "tempo"]

for nombre, df in carreras.items():
    sexo_ok = df["posto_sexo"].map(lambda t: parsear_posto_sexo(t)[0]).notna()
    raros = df.loc[~sexo_ok, columnas_seguras]
    if len(raros) > 0:
        print(nombre)
        print(raros.to_string(index=False))
        print()

carrera_01
posto posto_sexo posto_categoria    tempo
  638          —            NO-1 00:34:33
  860          —            NO-2 00:37:23
  876          —            NO-3 00:37:38

carrera_02
posto posto_sexo posto_categoria    tempo
  112          —            NO-1 00:28:35
  262          —            NO-2 00:32:03
  511          —            NO-3 00:36:13
  518          —            NO-4 00:36:21
  818          —            NO-5 00:43:20
  823          —            NO-6 00:43:26

carrera_03
posto posto_sexo posto_categoria    tempo
   87          —            NO-1 00:28:13
  266          —            NO-2 00:32:26
  307          —            NO-3 00:33:11
  327          —            NO-4 00:33:35
  454          —            NO-5 00:36:04

carrera_05
posto posto_sexo posto_categoria    tempo
  657          —            NO-1 10:28:43
  658          —            NO-2 10:39:57
  659          —            NO-3 10:42:58



### Tiempos imposibles en Volta a Oza
carrera_05 no tiene regulamento, así que se usan los límites generales (12 min y 1 h).

In [8]:
segundos_05 = carreras["carrera_05"]["tempo"].map(tiempo_a_segundos)
marca_05 = segundos_05.map(lambda s: marcar_tiempo(s, distancias.get("carrera_05")))
carreras["carrera_05"].loc[marca_05 == "imposible", columnas_seguras]

,posto,posto_sexo,posto_categoria,tempo
656,657,—,NO-1,10:28:43
657,658,—,NO-2,10:39:57
658,659,—,NO-3,10:42:58


### Categorías que aparecen
Para confirmar los códigos que faltaban (SUB20, MS3, discapacidad).

In [9]:
todas = pd.concat(carreras.values())
todas["posto_categoria"].map(lambda t: parsear_posto_categoria(t)[0]).value_counts(dropna=False)

posto_categoria
SNR      1667
MS1      1382
MS2      1328
MS3       590
SUB23     186
MS4       145
SUB20      72
NaN        17
ESP         3
Name: count, dtype: int64

## Conclusiones del perfil
- **Datos muy limpios en general**: sin vacíos, sin duplicados, sin postos ni dorsales repetidos, y el sexo siempre coincide con el de la categoría.
- **17 filas sin categoría ni sexo** (3 en carrera_01, 6 en carrera_02, 5 en carrera_03, 3 en carrera_05): vienen como `NO-1`, `NO-2`... en posto_categoria y `—` en posto_sexo. En la limpieza pasan a `SIN_CATEGORIA` con sexo vacío; cuentan para la participación pero no para los análisis por sexo o categoría.
- **3 categorías especiales `ESP`** (carrera_01, 05 y 06): se mantienen como categoría propia.
- **3 tiempos imposibles en Volta a Oza** (puestos 657-659, más de 10 h): son 3 de las filas `NO`. Parece que se registró la hora de llegada (la carrera salió a las 10:00) en vez del tiempo, pero no se puede comprobar: se marcan como `imposible` y no se corrigen.

- **Códigos confirmados**: aparecen SUB20 (72 filas) y MS3 (590), como se había deducido. No aparece ninguna categoría de discapacidad.

## Limpieza
Se aplica `limpiar_carrera()` de `src/limpieza.py` a cada carrera y se guarda en `datos/interim/carrera_XX_limpia.csv` (no se sube al repo: lleva nombres).

In [10]:
from src.limpieza import limpiar_carrera

limpias = {}
for nombre, df in carreras.items():
    limpia = limpiar_carrera(df, nombre, distancias.get(nombre))
    # Comprobación: no se pierde ni se duplica ninguna fila
    assert len(limpia) == len(df)
    limpia.to_csv(DIR_INTERIM / f"{nombre}_limpia.csv", index=False)
    limpias[nombre] = limpia
    print(nombre, len(limpia), "filas ->", f"{nombre}_limpia.csv")

carrera_01 1489 filas -> carrera_01_limpia.csv
carrera_02 929 filas -> carrera_02_limpia.csv
carrera_03 795 filas -> carrera_03_limpia.csv
carrera_04 875 filas -> carrera_04_limpia.csv
carrera_05 659 filas -> carrera_05_limpia.csv
carrera_06 643 filas -> carrera_06_limpia.csv


### Resumen de la limpieza (solo recuentos)

In [11]:
resumen = pd.concat(limpias.values())
print(resumen["tempo_marca"].value_counts(), "\n")
print(resumen["categoria"].value_counts())

tempo_marca
ok           5387
imposible       3
Name: count, dtype: int64 

categoria
SNR              1667
MS1              1382
MS2              1328
MS3               590
SUB23             186
MS4               145
SUB20              72
SIN_CATEGORIA      17
ESP                 3
Name: count, dtype: int64
